# SAKE Classical ML Master Notebook

This notebook is to: 
- Generate the instructions to run Swiss Army Knife (SAKE) for T1-weighted MRI preprocessing using FSL, SPM12, CAT12, Freesurfer, SynthSeg and/or Fastsurfer
- Combine SAKE's outputs to generate imaging feature matrices
- Run Classical Machine Learning (ML) training and testing, including SHAP generation

## Setup
Always run the below cell first before any of the other cells!

In [ ]:
import pandas as pd
import numpy as np
import os
import shutil
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import math
import glob

from SAKE_functions import *

wd          = "./SAKE_scripts/classical_ML" # REPLACE with your full working directory path
sakepath    = "./SAKE_scripts/sake/sake.sh"
outdir      = "./SAKE_outputs" # REPLACE with a path for all the preprocessing outputs
MASTERpath  = os.path.join(datadir, "SAKE_MASTER.csv")

subinfo     = pd.read_csv(MASTERpath)

pipelines   = ["fsl", "spm", "cat12", "freesurfer", "synthseg", "fastsurfer"]
models      = ["SVM", "RF", "LR", "NB", "kNN", "XGB", "MLP"]
featfilters = ["mrmr", "nofilt", "l1"]
seeds       = [38, 39, 40, 41, 42]

---

## Preprocessing - generating feature matrices

### Run SAKE
Generate the instructions to run SAKE preprocessing on the desired T1-weighted MRIs. This also checks for completed runs, to avoid re-running participant MRIs.

**NOTE**: This produces `commands.txt`. You can either run each line from `commands.txt` one by one from your terminal's command line, or you can submit them for running in parallel.

**NOTE**: This uses `SAKE_MASTER.csv`, which is a data summary csv containing, for each participant's T1-weighted MRI, their: 
- Unique identifier ('PTID')
- Sex ('PTGENDER', 0 = Female, 1 = Male)
- Age ('T1AGE')
- Diagnosis ('DIAGNOSIS', 0 = control, 1 = Alzheimer's Disease)
- The full path to their MRI nifti file ('T1_path', format example: '/path/to/images/mri.nii.gz')

In [ ]:
cmd = []
missing = []
pipelinecheck = []
suffixes = ["-fsl", "-spm", "-cat", "-fs", "-ss", "-fts"]

print('Making HPC commands...', end='')
for idx in range(subinfo.shape[0]):
    sub = subinfo.iloc[idx]["PTID"]
    date = str(subinfo.iloc[idx]["EXAMDATE_T1"])[0:10]
    t1path = subinfo.iloc[idx]["T1_path"]
    
    if t1path == -4:
        missing.append(f"{sub},{date},-4")
    elif not os.path.isfile(t1path):
        missing.append(f"{sub},{date},{t1path}")
    else:
        ses = t1path.split('/')[-2]
        fulloutdir = os.path.join(outdir, sub, ses)
        checks = []
        for folder, suffix in zip(pipelines, suffixes):
            matching_files = glob.glob(os.path.join(fulloutdir, folder, "*_fx.csv")) #check to see if fx.csv already made)
            if not matching_files:
                cmd.append(f"{sakepath} -i {t1path} -o {fulloutdir} {suffix}")
                checks.append(0)
            else:
                csv = pd.read_csv(matching_files[0])
                if folder == "cat12" and csv.isna().sum().sum() > 9: #remove cat12 outputs missing over 9 features (9 columns are expected to be blank)
                    shutil.rmtree(os.path.join(fulloutdir, folder))
                    checks.append(0)
                elif folder != "cat12" and csv.isna().sum().sum() > 0: #remove other outputs missing any feature
                    shutil.rmtree(os.path.join(fulloutdir, folder))
                    checks.append(0)
                else:
                    checks.append(1)                
        pipelinecheck.append(f"{sub},{ses}," + ",".join(map(str, checks)))
print('Done!')

#Print out the commands and pipeline checks into txt and csv files  
with open(os.path.join(wd, "missingT1.csv"), "w") as file:
    file.write("PTID,EXAMDATE_T1\n")
    file.write("\n".join(missing) + "\n")

with open(os.path.join(wd, "pipelinecheck.csv"), "w") as file:
    file.write("PTID,EXAMDATE_T1,FSL,SPM,CAT12,FS,SS,FTS\n")
    file.write("\n".join(pipelinecheck) + "\n")
        
with open(os.path.join(wd, "commands.txt"), "w") as file:
    file.write("\n".join(cmd) + "\n")
        
print('Please find commands.txt, missingT1.csv and pipelinecheck.csv in ' + wd)

---

### Combine feature csv files

Amalgamate the features for each pipeline into a full feature matrix csv.

In [ ]:
# Amalgamate the features of all subjects for a pipeline into a full raw feature matrix
raw_fx('FSL_fx.csv', "fsl", outdir, subinfo, wd)
raw_fx('SPM_fx.csv', "spm", outdir, subinfo, wd)
raw_fx('CAT12_fx.csv', "cat12", outdir, subinfo, wd)
raw_fx('FS_fx.csv', "freesurfer", outdir, subinfo, wd)
raw_fx('SS_fx.csv', "synthseg", outdir, subinfo, wd)
raw_fx('FTS_fx.csv', "fastsurfer", outdir, subinfo, wd)

### Data preparation

Combine the feature matrices with demographic information.

In [ ]:
# Produce a clean feature matrix for each pipeline
pipelines  = ["fsl", "spm", "cat12", "freesurfer", "synthseg", "fastsurfer"]

for pipeline in pipelines:
    print(f"Preparing fx.csv for {pipeline}...")
    # Find and check if raw_fx.csv exists for this pipeline
    csv_find = glob.glob(os.path.join(wd, pipeline, "raw_fx.csv"))
    if csv_find:
        fx = pd.read_csv(csv_find[0])
    else:
        print("No raw_fx.csv found.")

    # Remove empty and near empty (missing 80%+) columns & rows
    missing_fraction = fx.isna().mean()
    cols_to_drop = missing_fraction[missing_fraction >= 0.8].index.tolist()
    print("Empty or near empty columns:", cols_to_drop)
    fx = fx.drop(columns=cols_to_drop)

    rows_with_nan_mask = fx.isna().any(axis=1)
    ptids_with_nan = fx.loc[rows_with_nan_mask, "PTID"]
    print("Rows with empty values:", ptids_with_nan.tolist())
    print("Number of rows dropped:", {rows_with_nan_mask.sum()})
    fx = fx.loc[~rows_with_nan_mask].copy()
    
    # Check for any remaining NaNs
    print("Total number of missing values in the matrix:", fx.isna().sum().sum())

    # Merge with demographic data from subinfo
    fx = pd.merge(subinfo, fx, on='PTID', how='inner')
    
    # Save fx to csv
    fx.to_csv(os.path.join(wd, pipeline, 'fx.csv'), index=False)
    print(f"fx saved to: {os.path.join(wd, pipeline, 'fx.csv')}")
    print("-------------------------------------")

---

## Classical ML

### Classical ML

Train and test the classical ML models on all pipeline features.

In [ ]:
# Run a single preprocessing-model-feature selection combination
run_ML(wd, "fsl", "SVM", featfilter="l1", seed=42)

In [ ]:
# Run all combinations
for pipeline in pipelines:
    for seed in seeds:
        for model in models:
            for featfilter in featfilters:
                run_ML(wd, pipeline, model, featfilter=featfilter, seed=seed)

In [ ]:
# Concatenate csv performance files
columns = ['accuracy', 'precision', 'sensitivity', 'specificity', 'f1', 'auc_roc', 'optimal_threshold']
all_data = []
for pipeline in pipelines:
    for seed in seeds:
        for model in models:
            for featfilter in featfilters:
                file_path = os.path.join(wd, pipeline, str(seed), model, featfilter, f"{model}_{pipeline}_performance_df.csv")
                if os.path.exists(file_path):
                    df = pd.read_csv(file_path)
                    df['Pipeline']   = pipeline
                    df['Seed']       = seed
                    df['Model']      = model
                    df['FeatFilter'] = featfilter
                    all_data.append(df)
                else:
                    print(f"File not found: {file_path}")

combined_df = pd.concat(all_data, ignore_index=True)
combined_df = combined_df[['Model', 'Seed', 'Pipeline', 'FeatFilter'] + columns]
combined_df.to_csv(os.path.join(wd, "combined_performance_df.csv"), index=False)

# Produce mean and std values across seeds
summary = (combined_df.groupby(['Model', 'Pipeline', 'FeatFilter'])[columns].agg(['mean', 'std']))
summary.columns = [f"{metric}_{stat}" for metric, stat in summary.columns]
summary = summary.reset_index()
summary.to_csv(os.path.join(wd, "combined_performance_df_means-across-seeds.csv"), index=False)
summary.head()

---

## Post-hoc analyses

### Statistical testing
Using bootstrap CIs and Cohen's d.

Cannot use a standard test (e.g., Wilcoxon) as we only had 5 observations (seeds) per combination, meaning the minimum achievable p-value is capped (impossible to reach p<0.05) regardless of effect size.

**Bootstrap CIs** sidestep this by directly estimating the sampling distribution of the mean through resampling with replacement (e.g., 10,000 iterations). Gives an uncertainty estimate for each combination's performance without assumptions about the underlying distribution, and non-overlapping 95% CIs provide a conservative criterion for identifying performance differences (roughly equivalent to p<0.01 for a two-sided test).

**Cohen's d** quantifies the magnitude of differences between pairs of combinations in units of pooled standard deviation, independently of sample size. Important here as even where CIs overlap - meaning we lack the statistical power to declare a significant difference - we can still characterise whether the gap between two combinations is negligible (|d|<0.2), moderate (0.2–0.8), or large (|d|≥0.8). Allows us to make substantive claims about practical equivalence among top-tier combinations and categorical separation of SPM/NB, without overstating certainty.

**TLDR** bootstrap CIs answer "are these different?", Cohen's d answers "by how much?".

In [ ]:
# Stats - bootstrap CI and Cohen's d
import os
import pandas as pd
import numpy as np
from itertools import combinations

np.random.seed(42)
N_BOOTSTRAP = 10000
ALPHA       = 0.05
metrics     = ["accuracy", "auc_roc", "sensitivity", "specificity", "precision", "f1"]

# -----------------------------------------------------------------------
# Load per-seed ADNI data
# -----------------------------------------------------------------------
adni_df = pd.read_csv(os.path.join(wd, "combined_performance_df.csv"))
adni_df["combo"] = adni_df["Model"] + "+" + adni_df["Pipeline"] + "+" + adni_df["FeatFilter"]

combos = sorted(adni_df["combo"].unique())
seeds  = sorted(adni_df["Seed"].unique())
print(f"Combinations: {len(combos)} | Seeds: {len(seeds)}")

# -----------------------------------------------------------------------
# Core functions
# -----------------------------------------------------------------------
def bootstrap_ci(values, n_bootstrap=N_BOOTSTRAP, alpha=ALPHA):
    values     = np.array(values)
    n          = len(values)
    boot_means = np.array([
        np.mean(np.random.choice(values, size=n, replace=True))
        for _ in range(n_bootstrap)
    ])
    return np.percentile(boot_means, 100 * alpha / 2), \
           np.percentile(boot_means, 100 * (1 - alpha / 2))

def cohens_d(a, b):
    a, b       = np.array(a), np.array(b)
    pooled_std = np.sqrt((np.std(a, ddof=1)**2 + np.std(b, ddof=1)**2) / 2)
    if pooled_std == 0:
        return 0.0
    return (np.mean(a) - np.mean(b)) / pooled_std

# -----------------------------------------------------------------------
# 1. Bootstrap CIs per combination per metric
# -----------------------------------------------------------------------
ci_records = []
for metric in metrics:
    pivot = adni_df.pivot_table(index="combo", columns="Seed", values=metric)
    for combo in combos:
        vals         = pivot.loc[combo].values.astype(float)
        lower, upper = bootstrap_ci(vals)
        ci_records.append({
            "metric"  : metric,
            "combo"   : combo,
            "mean"    : np.mean(vals),
            "std"     : np.std(vals, ddof=1),
            "ci_lower": lower,
            "ci_upper": upper,
            "ci_width": upper - lower,
        })

ci_df = pd.DataFrame(ci_records)
ci_df.to_csv(os.path.join(wd, "ADNI_bootstrap_CIs.csv"), index=False)

# -----------------------------------------------------------------------
# 2. Pairwise effect sizes (Cohen's d) + CI overlap for all pairs
# -----------------------------------------------------------------------
effect_records = []
combo_pairs    = list(combinations(combos, 2))
ci_lookup      = ci_df.set_index(["metric", "combo"])

for metric in metrics:
    print(f"Computing pairwise effects for: {metric}")
    pivot = adni_df.pivot_table(index="combo", columns="Seed", values=metric)

    for combo_a, combo_b in combo_pairs:
        vals_a = pivot.loc[combo_a].values.astype(float)
        vals_b = pivot.loc[combo_b].values.astype(float)
        d      = cohens_d(vals_a, vals_b)

        ci_a       = ci_lookup.loc[(metric, combo_a)]
        ci_b       = ci_lookup.loc[(metric, combo_b)]
        ci_overlap = not (ci_a["ci_upper"] < ci_b["ci_lower"] or
                          ci_b["ci_upper"] < ci_a["ci_lower"])

        effect_records.append({
            "metric"    : metric,
            "combo_a"   : combo_a,
            "combo_b"   : combo_b,
            "cohens_d"  : d,
            "abs_d"     : abs(d),
            "ci_overlap": ci_overlap,
        })

effects_df = pd.DataFrame(effect_records)
effects_df.to_csv(os.path.join(wd, "ADNI_pairwise_effectsizes.csv"), index=False)

# -----------------------------------------------------------------------
# 3. Summary
# -----------------------------------------------------------------------
print("\n--- Bootstrap CI widths (median across combinations) ---")
print(ci_df.groupby("metric")["ci_width"].median().round(4).to_string())

print("\n--- Effect size summary (all pairs) ---")
for metric in metrics:
    sub          = effects_df[effects_df["metric"] == metric]
    n_nonoverlap = (~sub["ci_overlap"]).sum()
    n_total      = len(sub)
    median_d     = sub["abs_d"].median()
    large_d      = (sub["abs_d"] >= 0.8).sum()
    print(f"{metric:12s}: median |d|={median_d:.2f} | "
          f"non-overlapping CIs: {n_nonoverlap}/{n_total} "
          f"({100*n_nonoverlap/n_total:.1f}%) | "
          f"|d|≥0.8 (large): {large_d}/{n_total}")

# -----------------------------------------------------------------------
# 4. Per-pipeline and per-model effect size breakdown
#    (unbiased — all pipelines/models treated equally)
# -----------------------------------------------------------------------
print("\n--- Mean |d| by pipeline (vs all other pipelines) ---")
effects_df["pipeline_a"] = effects_df["combo_a"].str.split("+").str[1]
effects_df["pipeline_b"] = effects_df["combo_b"].str.split("+").str[1]

for metric in metrics:
    sub = effects_df[effects_df["metric"] == metric]
    print(f"\n  {metric}:")
    pipelines = sorted(set(effects_df["pipeline_a"].unique()))
    for pipe in pipelines:
        cross_pipe = sub[
            (sub["pipeline_a"] == pipe) != (sub["pipeline_b"] == pipe)
        ]
        print(f"    {pipe:12s}: median |d| vs others = {cross_pipe['abs_d'].median():.2f} "
              f"| non-overlapping CIs: {(~cross_pipe['ci_overlap']).sum()}/{len(cross_pipe)}")

In [ ]:
# Break down effect sizes by model vs pipeline to see what's driving differences
print("\n--- Mean |d| by MODEL (vs all other models) ---")
effects_df["model_a"] = effects_df["combo_a"].str.split("+").str[0]
effects_df["model_b"] = effects_df["combo_b"].str.split("+").str[0]

for metric in metrics:
    sub = effects_df[effects_df["metric"] == metric]
    print(f"\n  {metric}:")
    models = sorted(set(effects_df["model_a"].unique()))
    for model in models:
        cross_model = sub[
            (sub["model_a"] == model) != (sub["model_b"] == model)
        ]
        within_model = sub[
            (sub["model_a"] == model) & (sub["model_b"] == model)
        ]
        print(f"    {model:12s}: median |d| vs others = {cross_model['abs_d'].median():.2f} "
              f"| within same model = {within_model['abs_d'].median():.2f} "
              f"| non-overlapping CIs vs others: {(~cross_model['ci_overlap']).sum()}/{len(cross_model)}")

# Also check: among non-SPM only, what does the picture look like?
print("\n--- Non-SPM pairs only: effect size summary ---")
nonspm_effects = effects_df[
    ~effects_df["combo_a"].str.contains("spm") & 
    ~effects_df["combo_b"].str.contains("spm")
]
for metric in metrics:
    sub          = nonspm_effects[nonspm_effects["metric"] == metric]
    n_nonoverlap = (~sub["ci_overlap"]).sum()
    n_total      = len(sub)
    median_d     = sub["abs_d"].median()
    large_d      = (sub["abs_d"] >= 0.8).sum()
    print(f"{metric:12s}: median |d|={median_d:.2f} | "
          f"non-overlapping CIs: {n_nonoverlap}/{n_total} "
          f"({100*n_nonoverlap/n_total:.1f}%) | "
          f"|d|≥0.8 (large): {large_d}/{n_total}")

In [ ]:
# Non-SPM, non-NB pairs only
print("\n--- Non-SPM, non-NB pairs only: effect size summary ---")
top_effects = effects_df[
    ~effects_df["combo_a"].str.contains("spm") & 
    ~effects_df["combo_b"].str.contains("spm") &
    ~effects_df["combo_a"].str.startswith("NB") &
    ~effects_df["combo_b"].str.startswith("NB")
]
for metric in metrics:
    sub          = top_effects[top_effects["metric"] == metric]
    n_nonoverlap = (~sub["ci_overlap"]).sum()
    n_total      = len(sub)
    median_d     = sub["abs_d"].median()
    large_d      = (sub["abs_d"] >= 0.8).sum()
    print(f"{metric:12s}: median |d|={median_d:.2f} | "
          f"non-overlapping CIs: {n_nonoverlap}/{n_total} "
          f"({100*n_nonoverlap/n_total:.1f}%) | "
          f"|d|≥0.8 (large): {large_d}/{n_total}")

# And within-kNN check — kNN has high cross-model d but low within-model d
# suggesting it's a consistent moderate performer, not noisy
print("\n--- kNN within-pipeline breakdown (to check if it's consistently weaker) ---")
for metric in ["auc_roc", "accuracy"]:
    pivot = adni_df.pivot_table(index="combo", columns="Seed", values=metric)
    knn_combos = [c for c in combos if c.startswith("kNN")]
    top_combos = adni_df.groupby("combo")[metric].mean().nlargest(20).index.tolist()
    knn_in_top = [c for c in knn_combos if c in top_combos]
    print(f"{metric}: kNN combos in top 20 = {len(knn_in_top)}/21 kNN combos")
    print(f"  kNN mean {metric}: {adni_df[adni_df['combo'].str.startswith('kNN')][metric].mean()*100:.1f}%")
    print(f"  Non-kNN, non-NB, non-SPM mean {metric}: "
          f"{adni_df[~adni_df['combo'].str.startswith('kNN') & ~adni_df['combo'].str.startswith('NB') & ~adni_df['Pipeline'].str.contains('spm')][metric].mean()*100:.1f}%")